<a href="https://colab.research.google.com/github/castrokelly/PPGIa/blob/main/seminario_dft_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pontifícia Universidade Católica do Paraná — PUCPR

**Aluna:** 40131556 - Kelly Christine Alvarenga de Castro (christine.kelly@pucpr.edu.br)  
**Mestrado / Turma:** PPGIa (404) 2026/02  
**Disciplina:** Fundamentos de Matemática Computacional (Turma ME)  
**Professor:** Dr. Vinícius Mourão Alves de Souza

# SEMINÁRIO - Discrete Fourier Transform and Linear Algebra

## Demonstração prática  

Este notebook verifica numericamente os três fatos de Álgebra Linear que sustentam a apresentação:

1. a **DFT é um produto matriz-vetor** $X = F_N\,x$, e a matriz $F_N$ é **unitária** (a menos do fator $\sqrt{N}$): a inversa é a transposta conjugada;
2. **toda matriz circulante é diagonalizada por $F_N$**, o que transforma convolução (filtragem) em produto elemento a elemento (produto de Hadamard) no domínio da frequência;
3. a **FFT** calcula o mesmo produto $F_N\,x$ com custo $O(N\log N)$ em vez de $O(N^2)$.

Em todas as etapas, a implementação feita a partir da definição é comparada com a função de referência do NumPy (`np.fft.fft`, `np.linalg.solve`, `np.linalg.eigvals`), no mesmo espírito das atividades anteriores da disciplina.

## 1. Bibliotecas e configuração

Importando penas o **NumPy**. Fixamos uma semente aleatória para que os vetores de teste sejam reprodutíveis.

In [1]:
import numpy as np
import time

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True, linewidth=120)

## 2. A matriz DFT construída a partir da definição

A Transformada Discreta de Fourier de um vetor $x \in \mathbb{C}^N$ é o vetor $X \in \mathbb{C}^N$ com

$$
X[k] = \sum_{n=0}^{N-1} x[n]\, \omega^{kn}, \qquad \omega = e^{-2\pi i/N}, \qquad k = 0,\dots,N-1 .
$$

Isso é exatamente o produto matriz-vetor $X = F_N\,x$, em que a **matriz de Fourier** tem entradas $(F_N)_{k,n} = \omega^{kn}$.
A convenção de sinal ($-2\pi i$) é a mesma adotada por `np.fft.fft`; alguns livros (Strang, Aggarwal) usam o sinal oposto na matriz de síntese, o que apenas troca $F$ por $\overline{F}$.

A função abaixo constrói $F_N$ com `np.outer`, a mesma operação vista na Week 03/04: a matriz de expoentes $kn$ é o produto externo do vetor $[0,1,\dots,N-1]$ por ele mesmo.

In [2]:
def matriz_dft(N):
    n = np.arange(N)
    expoentes = np.outer(n, n)                 # matriz N x N com os produtos k*n
    return np.exp(-2j * np.pi * expoentes / N)  # omega^(k*n), omega = e^(-2*pi*i/N)


F4 = matriz_dft(4)
print("Matriz de Fourier F_4 (omega = -i):")
print(F4)

# --- Propriedade 1: F* F = N I  (F/sqrt(N) é unitária) --------------------------
N = 8
F = matriz_dft(N)
I = np.eye(N)

print("\nF* F = N I ?", np.allclose(F.conj().T @ F, N * I))
U = F / np.sqrt(N)
print("U = F/sqrt(N) é unitária (U* U = I)?", np.allclose(U.conj().T @ U, I))
print("|det(U)| = 1 ?", np.isclose(abs(np.linalg.det(U)), 1.0))
print("F é simétrica (F = F^T)?", np.allclose(F, F.T), "| F é Hermitiana (F = F*)?", np.allclose(F, F.conj().T))
print("U^4 = I ? (aplicar a DFT quatro vezes devolve o vetor original)", np.allclose(np.linalg.matrix_power(U, 4), I))

# --- Propriedade 2: X = F x coincide com np.fft.fft --------------------------------
# Série de exemplo do livro-texto da disciplina (Aggarwal, 2020, Problem 2.11.1)
s = np.array([8, 6, 2, 3, 4, 6, 6, 5], dtype=float)

X_matriz = F @ s
X_numpy = np.fft.fft(s)

print("\nX = F s (produto matriz-vetor):")
print(X_matriz)
print("Coincide com np.fft.fft(s)?", np.allclose(X_matriz, X_numpy))

# --- Propriedade 3: a inversa é a transposta conjugada (dividida por N) ------------
s_reconstruido = (F.conj().T @ X_matriz) / N
print("\nReconstrução x = F* X / N recupera s?", np.allclose(s_reconstruido, s))
print("Coincide com np.fft.ifft?", np.allclose(s_reconstruido, np.fft.ifft(X_matriz)))

# --- Propriedade 4: Parseval (a transformação preserva a norma, a menos do fator N)
print("\n||X||^2 = N ||s||^2 ?", np.isclose(np.linalg.norm(X_matriz) ** 2, N * np.linalg.norm(s) ** 2))

# --- Entrada real => simetria Hermitiana X[N-k] = conj(X[k]) (metade dos coeficientes é redundante)
print("X[N-k] = conj(X[k]) para s real?", np.allclose(X_matriz[1:][::-1], np.conj(X_matriz[1:])))

Matriz de Fourier F_4 (omega = -i):
[[ 1.+0.j  1.+0.j  1.+0.j  1.+0.j]
 [ 1.+0.j  0.-1.j -1.-0.j -0.+1.j]
 [ 1.+0.j -1.-0.j  1.+0.j -1.-0.j]
 [ 1.+0.j -0.+1.j -1.-0.j  0.-1.j]]

F* F = N I ? True
U = F/sqrt(N) é unitária (U* U = I)? True
|det(U)| = 1 ? True
F é simétrica (F = F^T)? True | F é Hermitiana (F = F*)? False
U^4 = I ? (aplicar a DFT quatro vezes devolve o vetor original) True

X = F s (produto matriz-vetor):
[40.   +0.j     5.414+5.414j  4.   -4.j     2.586-2.586j  0.   -0.j     2.586+2.586j  4.   +4.j     5.414-5.414j]
Coincide com np.fft.fft(s)? True

Reconstrução x = F* X / N recupera s? True
Coincide com np.fft.ifft? True

||X||^2 = N ||s||^2 ? True
X[N-k] = conj(X[k]) para s real? True


## 3. Matrizes circulantes, autovetores e o teorema da convolução

Uma **matriz circulante** $C$ é definida por um único vetor $c \in \mathbb{C}^N$: cada coluna é a coluna anterior deslocada ciclicamente uma posição para baixo. O produto $C\,x$ é a **convolução circular** $c \circledast x$ — a operação de filtragem.

O resultado central é:

$$
C = F_N^{-1}\,\operatorname{diag}(F_N\,c)\,F_N ,
$$

ou seja, **as colunas de $F_N^{-1}$ são autovetores de qualquer circulante** e os autovalores são a DFT do vetor $c$. Consequências verificadas abaixo:

- **teorema da convolução:** $F(c \circledast x) = (Fc) \circ (Fx)$ (produto de Hadamard, visto na Week 02);
- **sistema linear $Cx = b$** resolvido sem eliminação de Gauss: $x = F^{-1}\big[(Fb) / (Fc)\big]$;
- **determinante** de $C$ é o produto dos autovalores, $\det C = \prod_k (Fc)[k]$ (Week 05).

In [3]:
def matriz_circulante(c):
    c = np.asarray(c, dtype=float)
    N = len(c)
    # coluna k = vetor c deslocado ciclicamente k posições (np.roll)
    return np.column_stack([np.roll(c, k) for k in range(N)])


# Filtro passa-baixa de 3 pontos (pesos 0.2, 0.6, 0.2), escrito como vetor circulante
c = np.array([0.6, 0.2, 0, 0, 0, 0, 0, 0.2])
C = matriz_circulante(c)

print("Matriz circulante C gerada por c =", c)
print(C)

F = matriz_dft(len(c))
F_inv = F.conj().T / len(c)          # inversa de F, sem np.linalg.inv
Lambda = np.diag(F @ c)              # autovalores = DFT de c

# --- Diagonalização: C = F^{-1} diag(F c) F ------------------------------------------
print("\nC = F^{-1} diag(F c) F ?", np.allclose(C, F_inv @ Lambda @ F))

# --- Autovalores: comparação com np.linalg.eigvals (ordenados para comparar) ----------
autovalores_dft = np.sort_complex(F @ c)
autovalores_numpy = np.sort_complex(np.linalg.eigvals(C))
print("Autovalores de C = DFT de c ?", np.allclose(autovalores_dft, autovalores_numpy))

# --- Teorema da convolução: F(Cx) = (Fc) o (Fx) --------------------------------------
x = np.array([8, 6, 2, 3, 4, 6, 6, 5], dtype=float)
y = C @ x                                  # filtragem = convolução circular
print("\nx filtrado (C x) =", y)
print("F(C x) = (F c) * (F x) ?", np.allclose(F @ y, (F @ c) * (F @ x)))

# --- Sistema linear C x = b resolvido via DFT (sem eliminação de Gauss) ----------------
b = np.random.randint(1, 10, size=len(c)).astype(float)
x_dft = np.fft.ifft(np.fft.fft(b) / np.fft.fft(c)).real
x_solve = np.linalg.solve(C, b)
print("\nSolução de C x = b via DFT coincide com np.linalg.solve?", np.allclose(x_dft, x_solve))

# --- Determinante = produto dos autovalores -------------------------------------------
det_dft = np.prod(F @ c)
print("det(C) = produto de (F c) ?", np.isclose(np.linalg.det(C), det_dft.real), "| parte imaginária residual:", abs(det_dft.imag) < 1e-10)

# --- C é singular exatamente quando algum coeficiente (F c)[k] é zero -------------------
# Exemplo: o filtro (0.25, 0.5, 0.25) anula a frequência mais alta (k = N/2) e, portanto, det(C) = 0.
c_singular = np.array([0.5, 0.25, 0, 0, 0, 0, 0, 0.25])
print("\nFiltro (0.25, 0.5, 0.25): (F c)[N/2] =", np.round((F @ c_singular)[len(c) // 2].real, 6),
      "| det(C) = 0 ?", np.isclose(np.linalg.det(matriz_circulante(c_singular)), 0))

Matriz circulante C gerada por c = [0.6 0.2 0.  0.  0.  0.  0.  0.2]
[[0.6 0.2 0.  0.  0.  0.  0.  0.2]
 [0.2 0.6 0.2 0.  0.  0.  0.  0. ]
 [0.  0.2 0.6 0.2 0.  0.  0.  0. ]
 [0.  0.  0.2 0.6 0.2 0.  0.  0. ]
 [0.  0.  0.  0.2 0.6 0.2 0.  0. ]
 [0.  0.  0.  0.  0.2 0.6 0.2 0. ]
 [0.  0.  0.  0.  0.  0.2 0.6 0.2]
 [0.2 0.  0.  0.  0.  0.  0.2 0.6]]

C = F^{-1} diag(F c) F ? True
Autovalores de C = DFT de c ? True

x filtrado (C x) = [7.  5.6 3.  3.  4.2 5.6 5.8 5.8]
F(C x) = (F c) * (F x) ? True

Solução de C x = b via DFT coincide com np.linalg.solve? True
det(C) = produto de (F c) ? True | parte imaginária residual: True

Filtro (0.25, 0.5, 0.25): (F c)[N/2] = 0.0 | det(C) = 0 ? True


## 4. Custo computacional: produto matriz-vetor $O(N^2)$ versus FFT $O(N \log N)$

Na Week 05 vimos que o produto matriz-vetor com uma matriz $N \times N$ densa custa $O(N^2)$ multiplicações (perspectiva por elemento com $J = 1$).
A FFT de Cooley–Tukey fatora $F_N$ em $\log_2 N$ matrizes esparsas, reduzindo o custo para cerca de $\tfrac{1}{2} N \log_2 N$ multiplicações complexas.

O experimento abaixo mede o tempo de `F @ x` (após construir $F$) e de `np.fft.fft(x)` para alguns valores de $N$ (potências de 2), mantendo o melhor de cinco execuções. Para $N$ pequeno, o custo fixo de chamada das funções domina e a razão fica próxima de 1; a diferença assintótica aparece à medida que $N$ cresce.

In [4]:
def melhor_tempo(funcao, repeticoes=5):
    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        funcao()
        tempos.append(time.perf_counter() - inicio)
    return min(tempos)


print(f"{'N':>6} | {'N^2':>10} | {'N/2 log2 N':>11} | {'F @ x (s)':>11} | {'np.fft (s)':>11} | {'razão':>7}")
print("-" * 72)

for N in [256, 1024, 4096]:
    x = np.random.rand(N)
    F = matriz_dft(N)

    t_matriz = melhor_tempo(lambda: F @ x)
    t_fft = melhor_tempo(lambda: np.fft.fft(x))

    assert np.allclose(F @ x, np.fft.fft(x))   # os dois caminhos dão o mesmo resultado

    print(f"{N:>6} | {N**2:>10} | {int(N/2*np.log2(N)):>11} | {t_matriz:>11.2e} | {t_fft:>11.2e} | {t_matriz/t_fft:>7.0f}x")

     N |        N^2 |  N/2 log2 N |   F @ x (s) |  np.fft (s) |   razão
------------------------------------------------------------------------
   256 |      65536 |        1024 |    3.14e-05 |    2.17e-05 |       1x
  1024 |    1048576 |        5120 |    1.10e-03 |    4.16e-05 |      26x
  4096 |   16777216 |       24576 |    3.09e-02 |    1.23e-04 |     250x


## 5. Ponte com a Atividade 5: rotação da letra A e invariância do espectro

Na Atividade 5, a letra **A** foi rotacionada pela matriz $M = \begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}$.
Se cada ponto $(x_n, y_n)$ do contorno for representado pelo número complexo $z_n = x_n + i\,y_n$, a mesma rotação é a multiplicação pelo escalar $e^{i\theta}$.

Como a DFT é **linear**, $F(e^{i\theta} z) = e^{i\theta}\,F(z)$: todos os coeficientes giram pelo mesmo ângulo e as **magnitudes** $|Z[k]|$ não mudam. Esse é o princípio dos *Fourier descriptors* (Zahn & Roskies, 1972), usados como descritores de forma invariantes à rotação — uma possibilidade de extração de características para contornos de desenhos, como os do Clock Drawing Test.

In [5]:
# Contorno externo da letra A (mesma matriz da Atividade 5)
A1 = np.array([
    [0.0, 1.4, 2.6, 4.0, 3.0, 2.6, 1.4, 1.0, 0.0],
    [0.0, 5.0, 5.0, 0.0, 0.0, 1.5, 1.5, 0.0, 0.0]
])

angulo = 10
theta = np.radians(angulo)
M = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

T1 = M @ A1                                   # rotação como na Atividade 5

z = A1[0] + 1j * A1[1]                        # contorno como sequência complexa
z_rotacionado = T1[0] + 1j * T1[1]

print("Rotação por M equivale a multiplicar z por e^{i theta}?", np.allclose(z_rotacionado, np.exp(1j * theta) * z))

Z = np.fft.fft(z)
Z_rotacionado = np.fft.fft(z_rotacionado)

print("F(e^{i theta} z) = e^{i theta} F(z) ?", np.allclose(Z_rotacionado, np.exp(1j * theta) * Z))
print("Magnitudes |Z[k]| invariantes à rotação?", np.allclose(np.abs(Z), np.abs(Z_rotacionado)))
print("\n|Z[k]| (letra A original):   ", np.abs(Z))
print("|Z[k]| (letra A rotacionada):", np.abs(Z_rotacionado))

Rotação por M equivale a multiplicar z por e^{i theta}? True
F(e^{i theta} z) = e^{i theta} F(z) ? True
Magnitudes |Z[k]| invariantes à rotação? True

|Z[k]| (letra A original):    [20.616  2.326  7.88   4.583  3.115  1.041  4.31  11.192 14.606]
|Z[k]| (letra A rotacionada): [20.616  2.326  7.88   4.583  3.115  1.041  4.31  11.192 14.606]


## Conclusão

Os experimentos confirmam numericamente, com tolerância de ponto flutuante (`np.allclose`), os três fatos apresentados no seminário:

1. a DFT é o produto pela matriz $F_N$; como $F_N^{*}F_N = N I$, a transformada inversa é a transposta conjugada dividida por $N$ e a norma é preservada (Parseval) — não é necessário resolver um sistema linear por eliminação;
2. qualquer matriz circulante satisfaz $C = F^{-1}\operatorname{diag}(Fc)\,F$: os autovetores são fixos (as colunas de $F^{-1}$), os autovalores são a DFT de $c$, a convolução vira produto de Hadamard e o sistema $Cx = b$ é resolvido por três FFTs;
3. a FFT produz exatamente o mesmo vetor $F_N x$ com custo $O(N\log N)$, o que se traduz em ganhos de ordens de grandeza já para $N$ na casa dos milhares.

A seção 5 mostra que a rotação estudada na Atividade 5 corresponde a um escalar unitário no plano complexo, e que a magnitude do espectro é invariante a essa transformação linear.

## Referências

- AGGARWAL, Charu C. **Linear Algebra and Optimization for Machine Learning: A Textbook**. Springer, 2020. Seção 2.11.1, *The Discrete Fourier Transform*, p. 89–90.
- COOLEY, James W.; TUKEY, John W. An algorithm for the machine calculation of complex Fourier series. **Mathematics of Computation**, v. 19, n. 90, p. 297–301, 1965.
- GRAY, Robert M. Toeplitz and circulant matrices: a review. **Foundations and Trends in Communications and Information Theory**, v. 2, n. 3, p. 155–239, 2006. Teorema 3.1.
- STRANG, Gilbert. **Introduction to Linear Algebra**. 5. ed. Wellesley-Cambridge Press, 2016. Cap. 9, *Complex Vectors and Matrices* (Fourier matrix e FFT).
- VAN LOAN, Charles. **Computational Frameworks for the Fast Fourier Transform**. SIAM, 1992.
- ZAHN, Charles T.; ROSKIES, Ralph Z. Fourier descriptors for plane closed curves. **IEEE Transactions on Computers**, v. C-21, n. 3, p. 269–281, 1972.
- SOUZA, Vinícius M. A. **Fundamentals of Computational Mathematics — Weeks 01–06**. PUCPR, 2026. Material da disciplina.